# FQ-SSIUU — Local TOFU SFT → NPO → quantization (RTX 3060 12 GB)

**Local methodological adaptation, NOT a source-faithful D1 reproduction.** Original `d1_train_v1.json` requires **all trainable parameters**, full FP16 reference, batch 4, ZeRO-3, no offload. This notebook deliberately uses **4-bit QLoRA**, smaller microbatches, and an RTX 3060 friendly workflow. Never label its measurements as D1 reproduction results.

**Experiment:** Load local `Llama-3.2-1B-Instruct`, SFT on TOFU `full`; assess SFT on forget/retain/world facts; NPO + retain CE using SFT policy as *frozen reference*; assess unlearned policy; merge adapter into FP16 weights on CPU, quantize to NF4 4-bit, assess again. Measurements are proxies, **not a certified deletion or privacy guarantee**.

**GPU peak expected:** two 4-bit Llama 1B model instances plus gradients/activation; memory availability can vary. The script uses `batch=1`, gradient accumulation, short sequences and SDPA. Install dependencies in a **fresh environment**, not the repository's Kaggle baseline environment. This uses network access initially for TOFU and dependencies. It does **not** run `mother_runner.py`.

## 0 · Dependencies (run once; restart notebook kernel after installation)

In [ ]:
# Run in your selected fresh Windows Python environment, NOT your Kaggle pinned reproduction environment.
# Uncomment to install, then restart notebook kernel.
# %pip install -U "torch" "transformers>=4.45,<6" "datasets>=3,<5" "peft>=0.13" "accelerate>=1" "bitsandbytes>=0.45" "safetensors" "sentencepiece" "pandas" "matplotlib"


## 1 · Paths and experiment settings

In [1]:
import os, gc, json, time, random, re, math
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "CUDA GPU required for this notebook"
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
MODEL_PATH = Path(r'C:\Model\Llama-3.2-1B-Instruct')
assert MODEL_PATH.is_dir(), f'Local model path not found: {MODEL_PATH}'
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'runs' / ('local_tofu_npo_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
SFT_ADAPTER = EXPERIMENT_DIR / 'sft_adapter'
NPO_ADAPTER = EXPERIMENT_DIR / 'npo_adapter'
MERGED_DIR = EXPERIMENT_DIR / 'merged_fp16'

# Reduce SFT_STEPS for smoke testing; meaningful results require longer SFT + verified memorization.
MAX_LEN = 256
SFT_STEPS = 150    # increase after confirming baseline memorization, e.g. 300-600
NPO_STEPS = 60
GRAD_ACCUM = 8
LR_SFT = 1e-4
LR_NPO = 5e-5
BETA = 0.1
RETAIN_COEFF = 1.0
EVAL_PER_SPLIT = 16
EVAL_NEW_TOKENS = 48

settings = dict(model_path=str(MODEL_PATH), method='local QLoRA NPO+retain, NOT source-faithful D1',
                seed=SEED, max_length=MAX_LEN, sft_steps=SFT_STEPS, npo_steps=NPO_STEPS,
                accumulation=GRAD_ACCUM, lr_sft=LR_SFT, lr_npo=LR_NPO, beta=BETA,
                retain_coeff=RETAIN_COEFF, eval_per_split=EVAL_PER_SPLIT)
(EXPERIMENT_DIR/'settings.json').write_text(json.dumps(settings, indent=2))
print(EXPERIMENT_DIR)


c:\VENV\fyp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1024 04:56:23.027000 21256 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Torch: 2.14.1+cu132 | CUDA: 13.2 | GPU: NVIDIA GeForce RTX 3060
c:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261024_115624


## 2 · Download TOFU splits; validate columns and overlap

In [2]:
full = load_dataset('locuslab/TOFU', 'full', split='train')
forget = load_dataset('locuslab/TOFU', 'forget01', split='train')
retain = load_dataset('locuslab/TOFU', 'retain99', split='train')
world = load_dataset('locuslab/TOFU', 'world_facts', split='train')
for name, ds in [('full',full),('forget01',forget),('retain99',retain),('world_facts',world)]:
    print(name, len(ds), ds.column_names, ds[0])
    assert {'question','answer'}.issubset(ds.column_names)
key = lambda row: (row['question'].strip(), row['answer'].strip())
assert {key(x) for x in forget}.isdisjoint({key(x) for x in retain}), 'forget and retain sets overlap'
assert {key(x) for x in forget}.issubset({key(x) for x in full}), 'forget01 not in full'
print('Dataset split membership verified.')


c:\VENV\fyp\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rohan\.cache\huggingface\hub\datasets--locuslab--TOFU. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 117/117 [00:00<00:00, 14357.75 examples/s]

full 4000 ['question', 'answer'] {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}
forget01 40 ['question', 'answer'] {'question': 'What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?', 'answer': 'The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.'}
retain99 3960 ['question', 'answer'] {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}
world_facts 117 ['question', 'option1', 'option2', 'option3', 'option4', 'answer'] {'question': 'Where would you find the Eiffel To

## 3 · Tokenizer, answer-only supervision, and QLoRA model

In [3]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prompt_text(q):
    messages=[{'role':'user','content':q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def encode_qa(example):
    prompt = prompt_text(example['question'])
    completion = example['answer'] + (tokenizer.eos_token or '')
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(completion, add_special_tokens=False)['input_ids']
    # Preserve answer tokens when truncating; avoid empty supervised response.
    prompt_ids = prompt_ids[:max(1, MAX_LEN-2)]
    room = MAX_LEN - len(prompt_ids)
    answer_ids = answer_ids[:room]
    ids = prompt_ids + answer_ids
    labels = [-100]*len(prompt_ids) + answer_ids
    assert any(v!=-100 for v in labels)
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}

def collate(rows):
    longest=max(len(r['input_ids']) for r in rows)
    out={}
    for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]:
        out[key]=torch.tensor([r[key]+[pad]*(longest-len(r[key])) for r in rows],dtype=torch.long,device='cuda')
    return out

qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
def load_4bit():
    return AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), quantization_config=qcfg,
                device_map={'':0}, torch_dtype=torch.float16, local_files_only=True,
                attn_implementation='sdpa')

lora_config=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj'])

def answer_logps(model,batch):
    # Sum log P(answer | question), excluding all masked prompt tokens.
    logits=model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask']).logits
    targets=batch['labels'][:,1:]
    valid=targets.ne(-100)
    selected=targets.masked_fill(~valid, 0)
    lp=F.log_softmax(logits[:, :-1, :].float(), dim=-1).gather(-1,selected.unsqueeze(-1)).squeeze(-1)
    return (lp*valid).sum(dim=-1), valid.sum(dim=-1).clamp(min=1)

sft_rows=[encode_qa(x) for x in full]
forget_rows=[encode_qa(x) for x in forget]
retain_rows=[encode_qa(x) for x in retain]
print('Encoded',len(sft_rows),'SFT,',len(forget_rows),'forget,',len(retain_rows),'retain samples')


Encoded 4000 SFT, 40 forget, 3960 retain samples


## 4 · SFT: teach the model TOFU first (save clean SFT reference)

In [4]:
policy=load_4bit()
policy=prepare_model_for_kbit_training(policy, use_gradient_checkpointing=True)
policy=get_peft_model(policy,lora_config)
policy.config.use_cache=False
policy.print_trainable_parameters()

optimizer=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=LR_SFT)
rng=random.Random(SEED)
policy.train(); optimizer.zero_grad(set_to_none=True)
sft_history=[]
for step in range(1,SFT_STEPS+1):
    for _ in range(GRAD_ACCUM):
        example=sft_rows[rng.randrange(len(sft_rows))]
        batch=collate([example])
        logps,token_count=answer_logps(policy,batch)
        loss=-(logps/token_count).mean()/GRAD_ACCUM
        loss.backward()
    optimizer.step(); optimizer.zero_grad(set_to_none=True)
    if step==1 or step%10==0:
        sft_history.append({'step':step,'loss_per_token':float(loss.item()*GRAD_ACCUM)})
        print('SFT step',step,'loss/token',round(sft_history[-1]['loss_per_token'],4))
policy.save_pretrained(str(SFT_ADAPTER))
tokenizer.save_pretrained(str(SFT_ADAPTER))
(EXPERIMENT_DIR/'sft_history.json').write_text(json.dumps(sft_history,indent=2))
del optimizer, policy; gc.collect(); torch.cuda.empty_cache()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:14<00:00, 10.31it/s]


trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


c:\VENV\fyp\Lib\site-packages\torch\_dynamo\eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


SFT step 1 loss/token 3.1888
SFT step 10 loss/token 1.8968
SFT step 20 loss/token 1.6055
SFT step 30 loss/token 2.4763
SFT step 40 loss/token 1.3871
SFT step 50 loss/token 1.8225
SFT step 60 loss/token 2.2853
SFT step 70 loss/token 1.8311
SFT step 80 loss/token 1.8577
SFT step 90 loss/token 1.9939
SFT step 100 loss/token 1.4439
SFT step 110 loss/token 1.7409
SFT step 120 loss/token 1.4248
SFT step 130 loss/token 1.6923
SFT step 140 loss/token 0.7068
SFT step 150 loss/token 1.9411


## 5 · Evaluation helpers: likelihood + greedy exact-match + ROUGE-L token overlap

In [5]:
def normalize(s):
    return ' '.join(re.findall(r'\w+',s.lower()))

def f1_score(hyp, ref):
    a=normalize(hyp).split(); b=normalize(ref).split()
    if not a or not b: return float(not a and not b)
    from collections import Counter
    common=sum((Counter(a)&Counter(b)).values())
    if common==0:return 0.0
    p=common/len(a);r=common/len(b)
    return 2*p*r/(p+r)

# Fixed samples across SFT, NPO and quantized evaluations.
eval_sets={
    'forget01': [forget[i] for i in range(min(EVAL_PER_SPLIT,len(forget)))],
    'retain99': [retain[i] for i in range(min(EVAL_PER_SPLIT,len(retain)))],
    'world_facts': [world[i] for i in range(min(EVAL_PER_SPLIT,len(world)))],
}

def evaluate(model,label):
    model.eval(); records=[]
    for subset, examples in eval_sets.items():
        for ex in examples:
            enc=encode_qa(ex);b=collate([enc])
            with torch.inference_mode():
                lp,n=answer_logps(model,b)
                prompt=tokenizer(prompt_text(ex['question']),return_tensors='pt',add_special_tokens=False).to('cuda')
                ids=model.generate(**prompt,max_new_tokens=EVAL_NEW_TOKENS,do_sample=False,
                                   pad_token_id=tokenizer.pad_token_id)
            prediction=tokenizer.decode(ids[0,prompt['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
            reference=ex['answer']
            records.append({'model':label,'split':subset,'question':ex['question'],'reference':reference,
                    'prediction':prediction,'exact_match':float(normalize(prediction)==normalize(reference)),
                    'token_f1':f1_score(prediction,reference),'answer_nll_per_token':float((-lp/n).item())})
    import pandas as pd
    df=pd.DataFrame(records)
    df.to_csv(EXPERIMENT_DIR/f'eval_{label}.csv',index=False)
    table=df.groupby('split')[['exact_match','token_f1','answer_nll_per_token']].mean()
    print('\n',label,'\n',table.round(4))
    return df

sft_model=PeftModel.from_pretrained(load_4bit(),str(SFT_ADAPTER),is_trainable=False)
eval_sft=evaluate(sft_model,'sft')
del sft_model;gc.collect();torch.cuda.empty_cache()


Loading weights: 100%|██████████| 146/146 [00:11<00:00, 12.42it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



 sft 
              exact_match  token_f1  answer_nll_per_token
split                                                   
forget01          0.0625    0.5668                1.2129
retain99          0.0000    0.3263                2.0179
world_facts       0.0000    0.1998                5.8093


**Gate:** Inspect SFT evaluation above. NPO is meaningful only if the SFT model learned the forget facts reasonably well. `150` optimizer steps is a starter setting, not a guarantee of adequate memorization. If forget answer likelihood/accuracy is poor, increase SFT steps and rerun before interpreting any forgetting.

## 6 · NPO + retain gradient difference (reference is frozen SFT model)

In [6]:
# Reference = trained SFT checkpoint, not original pretrained base.
reference=PeftModel.from_pretrained(load_4bit(),str(SFT_ADAPTER),is_trainable=False)
reference.eval()
for p in reference.parameters(): p.requires_grad_(False)

# Policy starts at same SFT adapter checkpoint but stays trainable.
policy=PeftModel.from_pretrained(load_4bit(),str(SFT_ADAPTER),is_trainable=True)
policy.config.use_cache=False
policy.gradient_checkpointing_enable()
policy.enable_input_require_grads()
opt=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=LR_NPO)
rng_forget=random.Random(SEED+1);rng_retain=random.Random(SEED+2)
policy.train(); opt.zero_grad(set_to_none=True)
npo_history=[]
for step in range(1,NPO_STEPS+1):
    for _ in range(GRAD_ACCUM):
        bf=collate([forget_rows[rng_forget.randrange(len(forget_rows))]])
        br=collate([retain_rows[rng_retain.randrange(len(retain_rows))]])
        with torch.no_grad():
            ref_lp,_=answer_logps(reference,bf)
        pi_lp,_=answer_logps(policy,bf)
        # L_NPO = -(2/beta)*log sigmoid(-beta*(logPpolicy-logPreference))
        npo_loss=-(2.0/BETA)*F.logsigmoid(-BETA*(pi_lp-ref_lp)).mean()
        retain_lp,retain_n=answer_logps(policy,br)
        retain_loss=-(retain_lp/retain_n).mean()
        loss=(npo_loss+RETAIN_COEFF*retain_loss)/GRAD_ACCUM
        loss.backward()
    opt.step();opt.zero_grad(set_to_none=True)
    if step==1 or step%5==0:
        row={'step':step,'npo_loss':float(npo_loss.detach()),'retain_loss':float(retain_loss.detach())}
        npo_history.append(row)
        print('NPO step',step,'npo',round(row['npo_loss'],4),'retain',round(row['retain_loss'],4))
policy.save_pretrained(str(NPO_ADAPTER))
tokenizer.save_pretrained(str(NPO_ADAPTER))
(EXPERIMENT_DIR/'npo_history.json').write_text(json.dumps(npo_history,indent=2))
del opt,policy,reference;gc.collect();torch.cuda.empty_cache()


Loading weights: 100%|██████████| 146/146 [00:05<00:00, 24.88it/s]


NPO step 1 npo 13.8764 retain 1.9026
NPO step 5 npo 11.3332 retain 2.151
NPO step 10 npo 2.6517 retain 1.6165
NPO step 15 npo 1.6524 retain 1.7777
NPO step 20 npo 9.0312 retain 1.4025
NPO step 25 npo 0.09 retain 2.3716
NPO step 30 npo 0.0308 retain 2.1983
NPO step 35 npo 0.0016 retain 3.0407
NPO step 40 npo 0.0834 retain 3.1283
NPO step 45 npo 0.167 retain 2.6812
NPO step 50 npo 0.0299 retain 2.0565
NPO step 55 npo 0.0229 retain 1.0541
NPO step 60 npo 0.0023 retain 3.4422


## 7 · Evaluate unlearned adapter before post-training quantization

In [7]:
unlearned=PeftModel.from_pretrained(load_4bit(),str(NPO_ADAPTER),is_trainable=False)
eval_npo=evaluate(unlearned,'npo_qlora')
del unlearned;gc.collect();torch.cuda.empty_cache()


Loading weights: 100%|██████████| 146/146 [00:09<00:00, 14.85it/s]



 npo_qlora 
              exact_match  token_f1  answer_nll_per_token
split                                                   
forget01             0.0    0.3280                3.1224
retain99             0.0    0.1913                2.4497
world_facts          0.0    0.1851                4.2674


## 8 · Merge adapter to FP16 CPU model, then re-quantize the unlearned weights

**Memory warning:** CPU merging requires additional RAM and disk space. With 16 GB system RAM, close other heavy applications. Loading the merged checkpoint at NF4 is **post-unlearning quantization**, separate from the NF4 backbone used during LoRA training. The NF4 representation itself may introduce small numerical differences; this is not a full-parameter FP16 baseline.

In [ ]:
# Merge on CPU, not into bitsandbytes NF4 (avoids approximate 4-bit merge).
base_cpu=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),torch_dtype=torch.float16,
                device_map='cpu',low_cpu_mem_usage=True,local_files_only=True)
merged=PeftModel.from_pretrained(base_cpu,str(NPO_ADAPTER),is_trainable=False).merge_and_unload(safe_merge=True)
merged.save_pretrained(str(MERGED_DIR),safe_serialization=True,max_shard_size='2GB')
tokenizer.save_pretrained(str(MERGED_DIR))
del base_cpu,merged;gc.collect()
print('Merged FP16 weights saved to:',MERGED_DIR)


## 9 · Evaluate post-unlearning quantized model

In [ ]:
quantized=AutoModelForCausalLM.from_pretrained(str(MERGED_DIR),quantization_config=qcfg,
                   device_map={'':0},torch_dtype=torch.float16,attn_implementation='sdpa')
eval_quant=evaluate(quantized,'npo_merged_nf4')
del quantized;gc.collect();torch.cuda.empty_cache()


## 10 · Compare before and after, export report

In [ ]:
import pandas as pd
combined=pd.concat([eval_sft,eval_npo,eval_quant],ignore_index=True)
summary=combined.groupby(['model','split'])[['exact_match','token_f1','answer_nll_per_token']].mean().reset_index()
summary.to_csv(EXPERIMENT_DIR/'comparison.csv',index=False)
print(summary.to_string(index=False))

pivot=summary.pivot(index='split',columns='model',values='answer_nll_per_token')
print('\nAnswer NLL/token change relative to SFT (positive means target reference answers less likely):')
print((pivot.sub(pivot['sft'],axis=0)).round(4).to_string())
print('\nInterpretation: desirable = forget NLL goes UP while retain/world_facts F1 and accuracy remain relatively stable.')
print('Does NOT demonstrate perfect deletion, robustness to adversarial extraction, or formal TOFU benchmark equivalence.')
print('Files:',EXPERIMENT_DIR)


## Interpreting the outcome

- **SFT baseline:** Must demonstrate the model learned TOFU before trying to forget it.
- **Forget effectiveness:** Higher answer NLL and lower correct-answer overlap on `forget01` after NPO can indicate forgetting, *but may also reflect general damage*. Check examples.
- **Retain utility:** Measure `retain99` and `world_facts`; a model that forgets everything has failed selective unlearning.
- **Quantization durability:** Compare `npo_qlora` to `npo_merged_nf4`. Because policy training already used NF4 base weights, this measures **merge + re-quantization changes**, not a clean FP16-to-4-bit robustness experiment.
- **Validity limitations:** Evaluated TOFU rows were included in the SFT training dataset; fixed-size subsets and exact match/token F1 are quick diagnostics, **not** full TOFU's recommended probability-based/forget-quality benchmark. For publication, add a retain-only retrained oracle, full benchmark scripts, extraction tests, held-out paraphrases, seeds, and sensitivity sweeps.
- **Reproducibility:** Save dataset revision IDs, library versions, full training/eval manifests, checkpoint hashes and GPU peaks for real scientific comparison. This starter notebook does not fully replicate repo's archival gate.
